# Patch info extraction

Builds the per-slide patch coordinate/label dictionary used by STAGE1 training.

- Uses the **zero-fold single train/test split**, specifically the **TNBC-filtered cohort**
  (`0_folds/TNBC_train_df.csv`) that STAGE1 actually trains on — a broader, non-TNBC-filtered
  split would include patients outside the paper's cohort definition.
- Reads ROI coordinates/predictions from `TIGER_training/ROI_sampling_all/{coords,predictions}`
  (produced by `TIGER_training/labeling_TIGER_inference.py`).
- Slides with no matching ROI output are skipped and reported.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm

REPO_DIR = os.path.abspath('.')

TRAIN_DF_PATH = os.path.join(REPO_DIR, '0_folds', 'TNBC_train_df.csv')
SLIDE_DIR = '/path/to/your/wsi'
ROI_COORDS_DIR = os.path.join(REPO_DIR, 'TIGER_training', 'ROI_sampling_all', 'coords')
ROI_PRED_DIR = os.path.join(REPO_DIR, 'TIGER_training', 'ROI_sampling_all', 'predictions')

OUT_PKL_PATH = os.path.join(REPO_DIR, 'patch_info_KBSMC_train_zero_filtered.pkl')

train_df = pd.read_csv(TRAIN_DF_PATH, encoding='cp949')
print('train_df:', train_df.shape)
print(train_df['Recur'].value_counts().sort_index())

train_df: (107, 42)
Recur
0.0    94
1.0    13
Name: count, dtype: int64


In [2]:
def resolve_slide_name(tube_label, slide_name_reals):
    """WSI filenames often have their first '_' replaced with '-' (e.g. '1_067_11' -> '1-067_11').
    If the tube label doesn't match a filename directly, try this substitution."""
    if tube_label in slide_name_reals:
        return tube_label
    if tube_label[0] == '1':
        parts = tube_label.split('_')
        return parts[0] + '-' + parts[1] + '_' + parts[2]
    return None


slide_names = list(train_df['tube label'])
slide_name_reals = [word[:-4] for word in os.listdir(SLIDE_DIR)]

patch_info = []
skipped, failed = [], []

for idx, slide in enumerate(tqdm(slide_names)):
    slide_name = resolve_slide_name(slide, slide_name_reals)

    if slide_name is None:
        print(f"SKIPPING {slide} (no matching slide file)")
        skipped.append(slide)
        continue

    coords_path = os.path.join(ROI_COORDS_DIR, slide_name + '.npy')
    preds_path = os.path.join(ROI_PRED_DIR, slide_name + '.npy')

    if not os.path.isfile(coords_path) or not os.path.isfile(preds_path):
        print(f"SKIPPING {slide_name} (missing ROI coords/predictions)")
        skipped.append(slide_name)
        continue

    try:
        coord_values = np.load(coords_path)
        tiger_preds = np.load(preds_path)
        recur_label = train_df[train_df['tube label'] == slide]['Recur'].item()

        for iid, coord in enumerate(coord_values):
            pred = tiger_preds[iid]
            patch_info.append({
                'slide_name': slide_name,
                'slide_idx': idx,
                'x': coord[0],
                'y': coord[1],
                'img_name': '{}_{}_{}.png'.format(slide, coord[0], coord[1]),
                'label': recur_label,
                'category': pred,
            })
    except Exception as e:
        print(f"FAILED {slide_name}: {e}")
        failed.append(slide_name)

print(f"\npatch_info entries: {len(patch_info)}")
print(f"slides skipped: {len(skipped)} -> {skipped}")
print(f"slides failed: {len(failed)} -> {failed}")

100%|██████████| 107/107 [00:00<00:00, 132.08it/s]

patch_info entries: 685588
slides skipped: 0 -> []
slides failed: 0 -> []


In [3]:
categories = [word['category'] for word in patch_info]
print('unique categories:', np.unique(categories))

labels = [word['label'] for word in patch_info]
print('unique labels:', np.unique(labels))

n_slides = len(set(word['slide_name'] for word in patch_info))
print(f'slides contributing patches: {n_slides} / {len(slide_names)}')

unique categories: [0 1 2 3]
unique labels: [0. 1.]
slides contributing patches: 107 / 107


In [4]:
with open(OUT_PKL_PATH, 'wb') as f:
    pickle.dump(patch_info, f)

print(f'Saved patch_info ({len(patch_info)} entries) to {OUT_PKL_PATH}')

Saved patch_info (685588 entries) to ./patch_info_KBSMC_train_zero_filtered.pkl
